# Phase 3: LLM Integration
This notebook demonstrates how to connect our existing retrieval system to a Large Language Model (LLM).

In [1]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(dotenv_path="../.env")


True

In [3]:
# Load environment variables from .env file

# Model & Storage Settings
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "all-MiniLM-L6-v2")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "pdf_chunks")
CHROMA_DB_PATH = os.getenv("CHROMA_DB_PATH", "../data/vector_store")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")

# 1. Initialize Vector Store
embedding_function = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_function,
    persist_directory=CHROMA_DB_PATH
)

# 2. Initialize LLM (Local Ollama in this case)
llm = Ollama(model=OLLAMA_MODEL)
print("System loaded!")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


System loaded!


### Step 1: Query the vector store

In [4]:
query = "What is the main topic of the document?"
results = vectorstore.similarity_search_with_score(query, k=3)

context_pieces = []
for doc, score in results:
    context_pieces.append(doc.page_content)

context_str = "\n\n".join(context_pieces)
print("Retrieved Context:\n", context_str[:200], "...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved Context:
 extraction, and corpus construction. (i) NLP Model Pre-training:
n n n
recurrence expression is:
We fine-tune an NLP model on the SciERC dataset [41] to
(n − 1) r m
n n capture scientific relationship ...


### Step 2: Generate Answer

In [5]:
template = """You are a helpful AI assistant. Use the following pieces of retrieved context to answer the user's question. 
If you don't know the answer based on the context, just say that you don't know. Do not make up information.

--- CONTEXT ---
{context}
---------------

User Question: {query}
Answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

# Create the chain and execute it
chain = prompt | llm
answer = chain.invoke({
    "context": context_str,
    "query": query
})

print("Answer:\n", answer)

OllamaEndpointNotFoundError: Ollama call failed with status code 404. Maybe your model is not found and you should pull the model with `ollama pull llama3`.